# Sprint 4 — Webinar 13 (Teórico) — Student Version
## Data Journey → Funnels → Cohorts con SQL

**Programa:** Data Analytics  
**Sprint:** 4  
**Duración:** 100 minutos  
**Modalidad:** teoría guiada + práctica acompañada en SQL Workbench  
**Motor SQL:** [SQL Workbench](https://sql-workbench.com/) (DuckDB en el navegador)

> Este cuaderno está diseñado para que tomes apuntes, completes definiciones, construyas queries paso a paso y registres tus hallazgos durante la sesión.

<div style="text-align: center">
    <img src="https://raw.githubusercontent.com/ljpiere/tpdata_python/main/images/w1s1_2.png" width="400">
</div>

## Agenda sugerida (100 minutos)

| Tiempo | Bloque | ¿Qué aprenderé? | Mis notas |
|-------:|--------|-----------------|----------|
| 0–5 | Apertura | Objetivos, contexto y setup | |
| 5–20 | Bloque 1 | Data Journey: concepto, etapas, métricas y calidad | |
| 20–40 | Bloque 2 | CTEs: sintaxis, ventajas y ejercicios progresivos | |
| 40–60 | Bloque 3 | Funnels de conversión | |
| 60–80 | Bloque 4 | Cohorts y retención | |
| 80–95 | Bloque 5 | Ejercicios integradores | |
| 95–100 | Cierre | Takeaways y siguientes pasos | |

**Pregunta guía de la sesión:**  
¿Cómo pasamos de una tabla de eventos “cruda” a un análisis útil de journey, conversión y retención?

## Objetivos de aprendizaje

Completa la última columna al final de la sesión.

| Objetivo | ¿Lo logré? | En mis palabras |
|---|---|---|
| Explicar qué es un Data Journey | ☐ Sí / ☐ Parcial / ☐ No | |
| Identificar etapas y métricas de un journey | ☐ Sí / ☐ Parcial / ☐ No | |
| Entender qué es una CTE y cuándo usarla | ☐ Sí / ☐ Parcial / ☐ No | |
| Construir un funnel simple con SQL | ☐ Sí / ☐ Parcial / ☐ No | |
| Calcular drop-off y conversión | ☐ Sí / ☐ Parcial / ☐ No | |
| Formar cohorts y leer una tabla de retención | ☐ Sí / ☐ Parcial / ☐ No | |
| Segmentar resultados por atributos de negocio | ☐ Sí / ☐ Parcial / ☐ No | |

## Setup: Motor SQL

Usaremos **[SQL Workbench](https://sql-workbench.com/)** directamente en el navegador.

Marca lo que ya verificaste:

- ☐ Abrí SQL Workbench
- ☐ Confirmé que el motor es **DuckDB**
- ☐ Entiendo que hoy copiaré bloques SQL del notebook al editor
- ☐ Sé que debo empezar por el **Bloque 0**
- ☐ Tengo claro que este notebook funciona como guía de trabajo y apuntes

### Registro rápido de problemas
| Bloque | Error encontrado | ¿Cómo lo corregí? | ¿Quedó resuelto? |
|---|---|---|---|
| | | | |
| | | | |

---
# Bloque 0 · Creación de tablas y datos de práctica

> **Ejecuta este bloque completo primero** en SQL Workbench. Después, registra las verificaciones y observaciones solicitadas.

### 0.1 Creación de tablas (DDL)

Copia este bloque en SQL Workbench y ejecútalo.

### 0.1 Creación de tablas (DDL)

```sql
-- =============================================
-- DDL: Tablas para Journey / Funnel / Cohorts
-- Compatible con DuckDB (SQL Workbench)
-- =============================================

-- Tabla de usuarios con atributos de segmentación
CREATE OR REPLACE TABLE users (
  user_id     INTEGER PRIMARY KEY,
  signup_ts   TIMESTAMP NOT NULL,
  plan        VARCHAR NOT NULL,     -- 'free' o 'paid'
  channel     VARCHAR NOT NULL,     -- 'organic', 'ads', 'referral', 'email'
  device      VARCHAR NOT NULL      -- 'web', 'android', 'ios'
);

-- Tabla de eventos tipo GA4 minimalista
CREATE OR REPLACE TABLE events (
  event_id    INTEGER PRIMARY KEY,
  user_id     INTEGER NOT NULL,
  event_name  VARCHAR NOT NULL,
  event_ts    TIMESTAMP NOT NULL,
  props       VARCHAR NOT NULL DEFAULT '{}'
);
```

### Después de ejecutar
Completa:

- Tabla de usuarios creada correctamente: ☐ Sí / ☐ No  
- Tabla de eventos creada correctamente: ☐ Sí / ☐ No  
- ¿Qué campos parecen importantes para segmentar usuarios? _______________________________
- ¿Qué campos parecen importantes para analizar comportamiento? ___________________________

### 0.2 Carga de datos (Seed)

Copia este bloque en SQL Workbench y ejecútalo.

### 0.2 Carga de datos (Seed)

```sql
-- =============================================
-- SEED: Usuarios (cohortes enero y febrero 2021)
-- =============================================
INSERT INTO users (user_id, signup_ts, plan, channel, device) VALUES
  (1,  '2021-01-02 09:10:00', 'free',  'organic',  'web'),
  (2,  '2021-01-03 14:20:00', 'paid',  'ads',      'ios'),
  (3,  '2021-01-05 08:05:00', 'free',  'referral', 'android'),
  (4,  '2021-01-10 18:42:00', 'free',  'organic',  'web'),
  (5,  '2021-01-12 11:11:00', 'paid',  'email',    'web'),
  (6,  '2021-02-01 09:15:00', 'free',  'ads',      'android'),
  (7,  '2021-02-05 16:00:00', 'paid',  'referral', 'ios'),
  (8,  '2021-02-09 10:30:00', 'free',  'organic',  'web'),
  (9,  '2021-02-12 21:45:00', 'free',  'email',    'android'),
  (10, '2021-02-15 07:50:00', 'paid',  'ads',      'web'),
  -- Usuarios extra para cohort de marzo
  (11, '2021-03-01 10:00:00', 'free',  'organic',  'web'),
  (12, '2021-03-03 12:30:00', 'paid',  'ads',      'ios'),
  (13, '2021-03-08 09:15:00', 'free',  'referral', 'android'),
  (14, '2021-03-12 14:00:00', 'paid',  'email',    'web'),
  (15, '2021-03-18 17:45:00', 'free',  'organic',  'android');

-- =============================================
-- SEED: Eventos
-- Flujo: session_start → view_item → add_to_cart → begin_checkout → purchase
-- =============================================

-- Enero (usuarios 1..5)
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
  (1,  1, 'session_start',  '2021-01-02 09:12:00', '{"utm":"organic"}'),
  (2,  1, 'view_item',      '2021-01-02 09:14:00', '{"sku":"A101"}'),
  (3,  1, 'add_to_cart',    '2021-01-02 09:15:00', '{"sku":"A101"}'),
  (4,  1, 'begin_checkout', '2021-01-02 09:16:00', '{"cart_value":29.9}'),
  (5,  1, 'purchase',       '2021-01-02 09:18:00', '{"order_id":"O-0001","value":29.9}'),

  (6,  2, 'session_start',  '2021-01-03 14:21:00', '{"utm":"ads"}'),
  (7,  2, 'view_item',      '2021-01-03 14:22:00', '{"sku":"B222"}'),
  (8,  2, 'add_to_cart',    '2021-01-03 14:23:00', '{"sku":"B222"}'),
  (9,  2, 'begin_checkout', '2021-01-03 14:24:00', '{"cart_value":58.0}'),
  (10, 2, 'purchase',       '2021-01-03 14:27:00', '{"order_id":"O-0002","value":58.0}'),

  (11, 3, 'session_start',  '2021-01-05 08:06:00', '{"utm":"referral"}'),
  (12, 3, 'view_item',      '2021-01-05 08:07:00', '{"sku":"C333"}'),
  (13, 3, 'add_to_cart',    '2021-01-05 08:08:00', '{"sku":"C333"}'),
  (14, 3, 'begin_checkout', '2021-01-05 08:09:00', '{"cart_value":105.0}'),
  -- Usuario 3 NO compra (abandona en checkout)

  (15, 4, 'session_start',  '2021-01-10 18:43:00', '{"utm":"organic"}'),
  (16, 4, 'view_item',      '2021-01-10 18:46:00', '{"sku":"D444"}'),
  -- Usuario 4 solo mira un producto

  (17, 5, 'session_start',  '2021-01-12 11:12:00', '{"utm":"email"}'),
  (18, 5, 'view_item',      '2021-01-12 11:13:00', '{"sku":"A101"}'),
  (19, 5, 'add_to_cart',    '2021-01-12 11:14:00', '{"sku":"A101"}');
  -- Usuario 5 abandona el carrito

-- Febrero (usuarios 6..10)
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
  (20, 6, 'session_start',  '2021-02-01 09:17:00', '{"utm":"ads"}'),
  (21, 6, 'view_item',      '2021-02-01 09:18:00', '{"sku":"E555"}'),
  -- Usuario 6 solo mira

  (22, 7, 'session_start',  '2021-02-05 16:01:00', '{"utm":"referral"}'),
  (23, 7, 'view_item',      '2021-02-05 16:02:00', '{"sku":"B222"}'),
  (24, 7, 'add_to_cart',    '2021-02-05 16:03:00', '{"sku":"B222"}'),
  (25, 7, 'begin_checkout', '2021-02-05 16:04:00', '{"cart_value":58.0}'),
  (26, 7, 'purchase',       '2021-02-05 16:05:00', '{"order_id":"O-0007","value":58.0}'),

  (27, 8, 'session_start',  '2021-02-09 10:31:00', '{"utm":"organic"}'),
  (28, 8, 'view_item',      '2021-02-09 10:32:00', '{"sku":"C333"}'),
  (29, 8, 'add_to_cart',    '2021-02-09 10:34:00', '{"sku":"C333"}'),
  -- Usuario 8 abandona el carrito

  (30, 9, 'session_start',  '2021-02-12 21:46:00', '{"utm":"email"}'),
  -- Usuario 9 solo inicia sesión

  (31, 10, 'session_start',  '2021-02-15 07:51:00', '{"utm":"ads"}'),
  (32, 10, 'view_item',      '2021-02-15 07:52:00', '{"sku":"A101"}'),
  (33, 10, 'add_to_cart',    '2021-02-15 07:53:00', '{"sku":"A101"}'),
  (34, 10, 'begin_checkout', '2021-02-15 07:54:00', '{"cart_value":29.9}'),
  (35, 10, 'purchase',       '2021-02-15 07:56:00', '{"order_id":"O-0010","value":29.9}');

-- Marzo (usuarios 11..15) — actividad más ligera
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
  (36, 11, 'session_start',  '2021-03-01 10:05:00', '{"utm":"organic"}'),
  (37, 11, 'view_item',      '2021-03-01 10:08:00', '{"sku":"A101"}'),
  (38, 11, 'add_to_cart',    '2021-03-01 10:10:00', '{"sku":"A101"}'),
  (39, 11, 'begin_checkout', '2021-03-01 10:12:00', '{"cart_value":29.9}'),
  (40, 11, 'purchase',       '2021-03-01 10:15:00', '{"order_id":"O-0011","value":29.9}'),

  (41, 12, 'session_start',  '2021-03-03 12:35:00', '{"utm":"ads"}'),
  (42, 12, 'view_item',      '2021-03-03 12:37:00', '{"sku":"B222"}'),

  (43, 13, 'session_start',  '2021-03-08 09:20:00', '{"utm":"referral"}'),
  (44, 13, 'view_item',      '2021-03-08 09:22:00', '{"sku":"C333"}'),
  (45, 13, 'add_to_cart',    '2021-03-08 09:24:00', '{"sku":"C333"}'),

  (46, 14, 'session_start',  '2021-03-12 14:05:00', '{"utm":"email"}'),
  (47, 14, 'view_item',      '2021-03-12 14:07:00', '{"sku":"D444"}'),
  (48, 14, 'add_to_cart',    '2021-03-12 14:09:00', '{"sku":"D444"}'),
  (49, 14, 'begin_checkout', '2021-03-12 14:11:00', '{"cart_value":45.0}'),
  (50, 14, 'purchase',       '2021-03-12 14:13:00', '{"order_id":"O-0014","value":45.0}'),

  (51, 15, 'session_start',  '2021-03-18 17:50:00', '{"utm":"organic"}'),
  (52, 15, 'view_item',      '2021-03-18 17:52:00', '{"sku":"E555"}'),
  (53, 15, 'add_to_cart',    '2021-03-18 17:54:00', '{"sku":"E555"}');

-- Eventos de retención: usuarios que regresan semanas después
INSERT INTO events (event_id, user_id, event_name, event_ts, props) VALUES
  -- Usuario 1 regresa en semana 1 y semana 3
  (54, 1, 'session_start', '2021-01-09 10:00:00', '{"utm":"direct"}'),
  (55, 1, 'view_item',     '2021-01-09 10:05:00', '{"sku":"F666"}'),
  (56, 1, 'session_start', '2021-01-23 15:00:00', '{"utm":"email"}'),

  -- Usuario 2 regresa en semana 2
  (57, 2, 'session_start', '2021-01-17 12:00:00', '{"utm":"ads"}'),
  (58, 2, 'view_item',     '2021-01-17 12:05:00', '{"sku":"A101"}'),
  (59, 2, 'purchase',      '2021-01-17 12:10:00', '{"order_id":"O-0002b","value":29.9}'),

  -- Usuario 5 regresa en semana 1
  (60, 5, 'session_start', '2021-01-19 09:00:00', '{"utm":"organic"}'),

  -- Usuario 7 regresa en semana 1 y semana 2
  (61, 7, 'session_start', '2021-02-12 11:00:00', '{"utm":"referral"}'),
  (62, 7, 'view_item',     '2021-02-12 11:05:00', '{"sku":"A101"}'),
  (63, 7, 'session_start', '2021-02-20 14:00:00', '{"utm":"direct"}'),

  -- Usuario 10 regresa en semana 1
  (64, 10, 'session_start', '2021-02-22 08:00:00', '{"utm":"ads"}'),
  (65, 10, 'view_item',     '2021-02-22 08:05:00', '{"sku":"B222"}'),

  -- Usuario 11 regresa en semana 1
  (66, 11, 'session_start', '2021-03-08 11:00:00', '{"utm":"organic"}'),
  (67, 11, 'view_item',     '2021-03-08 11:05:00', '{"sku":"D444"}');
```

### Registro de observación
| Elemento | ¿Qué representa? | Observación rápida |
|---|---|---|
| `users.plan` | | |
| `users.channel` | | |
| `users.device` | | |
| `events.event_name` | | |
| `events.props` | | |

### 0.3 Verificación rápida

Ejecuta estas consultas y registra el resultado.

### 0.3 Verificación rápida

Ejecuta estas consultas para confirmar que los datos se cargaron correctamente:

```sql
-- Verificar usuarios
SELECT COUNT(*) AS total_users FROM users;
-- Esperado: 15

-- Verificar eventos
SELECT COUNT(*) AS total_events FROM events;
-- Esperado: 67

-- Vista rápida de la distribución de eventos
SELECT event_name, COUNT(*) AS cantidad
FROM events
GROUP BY event_name
ORDER BY cantidad DESC;
```

### Mis resultados
| Consulta | Resultado obtenido | ¿Coincide con lo esperado? | Interpretación |
|---|---:|---|---|
| `COUNT(*)` de `users` | | ☐ Sí / ☐ No | |
| `COUNT(*)` de `events` | | ☐ Sí / ☐ No | |
| Distribución por `event_name` | | ☐ Sí / ☐ No | |

---
# Bloque 1 · ¿Qué es el Data Journey? (15 min)

## 1.1 Definición del Data Journey

Completa con tus palabras:

- Un **Data Journey** es: ________________________________________________
- Es importante porque permite: _________________________________________
- No basta con mirar métricas aisladas porque: ____________________________

### Idea central
Convierte la definición en una frase corta para recordarla:
____________________________________________________________  
____________________________________________________________

## 1.2 Etapas típicas de un Data Journey

En este caso trabajaremos con un funnel base de e-commerce:

`session_start → view_item → add_to_cart → begin_checkout → purchase`

### Completa la lógica de cada etapa
| Etapa | ¿Qué hace el usuario? | ¿Por qué importa? |
|---|---|---|
| `session_start` | | |
| `view_item` | | |
| `add_to_cart` | | |
| `begin_checkout` | | |
| `purchase` | | |

### Mini-reflexión
¿En cuál etapa crees que suele perderse más gente en un producto digital? ¿Por qué?
____________________________________________________________  
____________________________________________________________

## 1.3 Métricas del Data Journey

Completa la tabla durante la explicación.

| Métrica | ¿Qué mide? | Ejemplo de interpretación |
|---|---|---|
| Conversión | | |
| Drop-off | | |
| Retención | | |
| Frecuencia de uso | | |
| Tiempo entre eventos | | |

### Pregunta guía
¿Qué diferencia hay entre una métrica de conversión y una de retención?
____________________________________________________________

## 1.4 Campos clave en la base de datos de eventos

Relaciona cada campo con su función.

| Campo | ¿Para qué sirve? | Ejemplo en esta clase |
|---|---|---|
| `user_id` | | |
| `event_ts` | | |
| `event_name` | | |
| `props` | | |

### Nota personal
¿Cuál de estos campos te parece más crítico para reconstruir un journey?
____________________________________________________________

## 1.5 Validación de calidad de datos

Antes de analizar, debemos validar los datos.

### Checklist de calidad
- ☐ Hay una llave identificable por usuario
- ☐ Los timestamps tienen sentido cronológico
- ☐ Los nombres de eventos son consistentes
- ☐ No hay duplicados evidentes
- ☐ Las etapas del journey existen realmente en la data

### Registra dos riesgos de calidad que podrían dañar un funnel
1. ____________________________________________________________
2. ____________________________________________________________

### Pregunta corta
Si faltara `event_ts`, ¿qué análisis dejaría de ser confiable?
____________________________________________________________

---
# Bloque 2 · ¿Qué son las CTEs? (20 min)

## 2.1 Definición y motivación

Completa:

- Una **CTE** es: ________________________________________________
- Se define con la cláusula: _____________________________________
- Ayuda porque hace las consultas más: ____________________________

### CTE vs subconsulta
| Aspecto | CTE | Subconsulta anidada |
|---|---|---|
| Legibilidad | | |
| Reutilización | | |
| Debugging | | |
| Mantenimiento | | |

## 2.2 Sintaxis de las CTEs

### Plantilla base
Completa los huecos:

```sql
WITH nombre_cte AS (
    SELECT ...
    FROM ...
    WHERE ...
)
SELECT ...
FROM nombre_cte;
```

### Reglas que debo recordar
1. Una CTE existe solo durante __________________________________
2. Una CTE puede referenciar ___________________________________
3. Si quiero varias CTEs, debo __________________________________

### Mi ejemplo mínimo
Escribe aquí una mini-CTE inventada:
```sql
WITH __________________ AS (
    SELECT __________________
)
SELECT __________________
FROM __________________;
```

## 2.3 Ejercicios progresivos de CTEs

Antes de escribir SQL, completa la estrategia:

| Ejercicio | Tabla base | Filtro principal | ¿Necesita JOIN? | Resultado esperado |
|---|---|---|---|---|
| CTE-1 | | | | |
| CTE-2 | | | | |
| CTE-3 | | | | |
| CTE-4 | | | | |
| CTE-5 | | | | |

### Ejercicio CTE-1: Tu primera CTE (Nivel: ⭐ Básico)

**Consigna:** Crea una CTE que seleccione solo los usuarios con plan `'free'` y luego cuenta cuántos hay.

**Antes de escribir el SQL**
- Nombre de la CTE: ______________________
- Tabla base: _____________________________
- Filtro: _________________________________
- Métrica final: __________________________

**Borrador**
```sql
WITH __________________ AS (
    SELECT __________________
    FROM __________________
    WHERE __________________
)
SELECT __________________
FROM __________________;
```

**Registro**
- Resultado obtenido: ______________________
- ¿Coincidió con lo esperado?: ☐ Sí / ☐ No
- ¿Qué aprendí con este ejercicio?: ______________________________

In [ ]:
# Escribe aquí tu SQL de apoyo o apuntes personales

### Ejercicio CTE-2: Dos CTEs encadenadas (Nivel: ⭐ Básico)

**Consigna:** Crea una CTE con los usuarios `'paid'` y otra CTE con los eventos de tipo `'purchase'`. Luego, cruza ambas para encontrar qué usuarios de pago hicieron al menos una compra.

### Planeación
| Paso | ¿Qué haré? |
|---|---|
| CTE 1 | |
| CTE 2 | |
| SELECT final | |

**Esqueleto**
```sql
WITH cte_1 AS (
    SELECT __________________
    FROM __________________
    WHERE __________________
),
cte_2 AS (
    SELECT __________________
    FROM __________________
    WHERE __________________
)
SELECT __________________
FROM __________________
__________ JOIN __________________
ON __________________;
```

**Observaciones**
- ¿Por qué este caso se entiende mejor con CTEs que con subconsultas?
____________________________________________________________

### Ejercicio CTE-3: CTE que referencia otra CTE (Nivel: ⭐⭐ Intermedio)

**Consigna:** Construye un flujo de 3 pasos:
1. Filtra eventos de enero 2021.
2. Cuenta cuántos eventos tiene cada usuario.
3. Quédate solo con usuarios con 3 o más eventos.

### Traducción a SQL
| Paso lógico | Nombre posible de la CTE | ¿Qué devuelve? |
|---|---|---|
| Filtrar enero | | |
| Contar por usuario | | |
| Filtrar usuarios activos | | |

**Mi SQL**
```sql
WITH __________________ AS (
    SELECT *
    FROM __________________
    WHERE __________________
),
__________________ AS (
    SELECT __________________, COUNT(*) AS __________________
    FROM __________________
    GROUP BY __________________
),
__________________ AS (
    SELECT *
    FROM __________________
    WHERE __________________
)
SELECT *
FROM __________________
ORDER BY __________________ DESC;
```

**Resultado**
| ¿Qué usuarios salieron? | ¿Qué significa “activo” en este caso? |
|---|---|
| | |

### Ejercicio CTE-4: Reutilizar una CTE múltiples veces (Nivel: ⭐⭐ Intermedio)

**Consigna:** Crea una CTE `resumen_usuario` que calcule para cada usuario:
- total de eventos,
- si tiene al menos una compra.

Luego, compara compradores vs. no compradores.

### Diseño de la solución
| Columna a construir | ¿Cómo la calcularías? |
|---|---|
| `total_eventos` | |
| `es_comprador` | |
| `segmento` final | |

**Borrador**
```sql
WITH resumen_usuario AS (
    SELECT
        __________________,
        __________________ AS total_eventos,
        __________________ AS es_comprador
    FROM __________________
    GROUP BY __________________
)
SELECT
    __________________ AS segmento,
    __________________ AS num_usuarios,
    __________________ AS promedio_eventos
FROM __________________
GROUP BY __________________
ORDER BY __________________;
```

**Interpretación**
¿Los compradores parecen tener más interacción que los no compradores?
____________________________________________________________

### Ejercicio CTE-5: CTE con JOIN a tabla original (Nivel: ⭐⭐⭐ Avanzado)

**Consigna:** Identifica el primer evento de cada usuario y luego únelo con `users` para mostrar:
`user_id`, `plan`, `channel`, `primer_evento`, `fecha_primer_evento`.

### Puntos clave
- Función de ventana a usar: ______________________
- Partición: _____________________________________
- Orden: _________________________________________
- Filtro final para quedarme con el primer evento: __________________

**Plantilla**
```sql
WITH primer_evento AS (
    SELECT
        __________________,
        __________________,
        __________________,
        ROW_NUMBER() OVER (
            PARTITION BY __________________
            ORDER BY __________________
        ) AS rn
    FROM __________________
)
SELECT
    __________________,
    __________________,
    __________________,
    __________________ AS primer_evento,
    __________________ AS fecha_primer_evento
FROM __________________
INNER JOIN __________________
    ON __________________
WHERE __________________
ORDER BY __________________;
```

**Reflexión**
¿Por qué esta técnica es útil en análisis de journey?
____________________________________________________________

---
# Bloque 3 · Construcción de Funnels de conversión (20 min)

## 3.1 ¿Qué es un funnel?

Un funnel de conversión resume cuántos usuarios pasan por cada etapa del journey y cuántos se pierden en el camino.

### Nuestro funnel
`session_start → view_item → add_to_cart → begin_checkout → purchase`

### Antes de calcular
Completa:

- Usuario que entra al funnel = __________________________________
- Usuario que convierte = ________________________________________
- Drop-off significa _____________________________________________
- Tasa de conversión global significa _____________________________

### Ejercicio F-1: Funnel básico con CTEs (Nivel: ⭐ Básico)

**Consigna:** Construye un funnel para todos los datos. Usa una CTE por etapa para contar usuarios distintos.

### Planeación
| Etapa | Nombre de la CTE | ¿Qué debe devolver? |
|---|---|---|
| session | | |
| view | | |
| cart | | |
| checkout | | |
| purchase | | |

**Borrador**
```sql
WITH cte_session AS (
    SELECT DISTINCT __________________
    FROM __________________
    WHERE __________________
),
cte_view AS (
    SELECT DISTINCT __________________
    FROM __________________
    WHERE __________________
),
cte_cart AS (
    SELECT DISTINCT __________________
    FROM __________________
    WHERE __________________
),
cte_checkout AS (
    SELECT DISTINCT __________________
    FROM __________________
    WHERE __________________
),
cte_purchase AS (
    SELECT DISTINCT __________________
    FROM __________________
    WHERE __________________
)
SELECT
    (SELECT COUNT(*) FROM __________________) AS session_users,
    (SELECT COUNT(*) FROM __________________) AS view_users,
    (SELECT COUNT(*) FROM __________________) AS cart_users,
    (SELECT COUNT(*) FROM __________________) AS checkout_users,
    (SELECT COUNT(*) FROM __________________) AS purchase_users;
```

**Mis resultados**
| Etapa | Usuarios |
|---|---:|
| Session | |
| View item | |
| Add to cart | |
| Begin checkout | |
| Purchase | |

### Ejercicio F-2: Funnel con drop-off y conversión (Nivel: ⭐⭐ Intermedio)

**Consigna:** A partir del funnel anterior, calcula:
- drop-off entre etapas,
- tasa de conversión global de `session_start` a `purchase`.

### Fórmulas
- Drop-off entre etapa A y B = ______________________________
- Conversión global = ______________________________________

**Borrador**
```sql
WITH ... ,
counts AS (
    SELECT
        __________________ AS n_session,
        __________________ AS n_view,
        __________________ AS n_cart,
        __________________ AS n_checkout,
        __________________ AS n_purchase
)
SELECT
    *,
    __________________ AS drop_session_to_view,
    __________________ AS drop_view_to_cart,
    __________________ AS drop_cart_to_checkout,
    __________________ AS drop_checkout_to_purchase,
    ROUND(100.0 * __________________ / NULLIF(__________________, 0), 2) AS conversion_rate_pct
FROM counts;
```

**Lectura del resultado**
- Mayor cuello de botella: ______________________________
- Posible causa: ________________________________________

### Ejercicio F-3: Funnel filtrado por mes (Nivel: ⭐⭐ Intermedio)

**Consigna:** Repite el funnel solo para **febrero 2021**.

### Qué cambia respecto al ejercicio anterior
- Necesito una CTE base que filtre: ____________________________________
- Todas las etapas ahora salen de: _____________________________________

**Plantilla**
```sql
WITH base AS (
    SELECT *
    FROM __________________
    WHERE __________________
),
cte_session AS (
    SELECT DISTINCT user_id FROM base WHERE __________________
),
cte_view AS (
    SELECT DISTINCT user_id FROM base WHERE __________________
),
cte_cart AS (
    SELECT DISTINCT user_id FROM base WHERE __________________
),
cte_checkout AS (
    SELECT DISTINCT user_id FROM base WHERE __________________
),
cte_purchase AS (
    SELECT DISTINCT user_id FROM base WHERE __________________
)
-- agrega aquí counts y métricas finales
```

**Comparación**
| Métrica | Funnel general | Funnel febrero | Diferencia / comentario |
|---|---:|---:|---|
| Session users | | | |
| Purchase users | | | |
| Conversión global | | | |

### 3.2 Interpretación de resultados del funnel

No basta con calcular el funnel: hay que traducirlo a negocio.

### Completa este análisis
| Pregunta | Mi respuesta |
|---|---|
| ¿Dónde está la mayor caída? | |
| ¿Qué hipótesis podría explicar esa caída? | |
| ¿Qué dato adicional pediría para validar la hipótesis? | |
| ¿Qué acción concreta propondría? | |

### Redacción corta para stakeholders
Escribe un mini hallazgo en 2–3 líneas:
____________________________________________________________  
____________________________________________________________

---
# Bloque 4 · Análisis de retención con Cohorts (20 min)

## 4.1 ¿Qué es un cohort?

Completa con tus palabras:

- Un **cohort** es un grupo de usuarios que comparten _______________________
- En esta clase el cohort se define por _________________________________
- La retención responde la pregunta _____________________________________

### Relación con el funnel
¿Cómo se complementan funnel y cohort?
____________________________________________________________  
____________________________________________________________

### Ejercicio C-1: Identificar cohorts mensuales (Nivel: ⭐ Básico)

**Consigna:** Muestra cada usuario con su cohort mensual y cuenta cuántos usuarios hay por cohort.

### Idea de la consulta
| Elemento | Respuesta |
|---|---|
| Tabla base | |
| Campo de fecha | |
| Función para truncar al mes | |
| Métrica final | |

**Borrador**
```sql
SELECT
    DATE_TRUNC('month', __________________) AS cohort_month,
    __________________ AS cohort_size
FROM __________________
GROUP BY __________________
ORDER BY __________________;
```

**Resultados**
| Cohort | Tamaño |
|---|---:|
| Enero | |
| Febrero | |
| Marzo | |

### Ejercicio C-2: Tabla de retención semanal (Nivel: ⭐⭐⭐ Avanzado)

**Consigna:** Construye una tabla de retención que muestre, para cada cohort mensual, el porcentaje de usuarios que tuvieron al menos un evento en las semanas 0, 1, 2 y 3.

### Descompón el problema
| Paso | ¿Qué debo construir? |
|---|---|
| 1 | Asignar cohort a cada usuario |
| 2 | Crear grid de semanas |
| 3 | Detectar si el usuario tuvo actividad en cada ventana |
| 4 | Calcular retenidos / tamaño del cohort |

**Conceptos a usar**
- `UNNEST(ARRAY[0,1,2,3])` sirve para: _______________________
- `DATEDIFF` sirve para: _____________________________________
- `EXISTS` sirve para: _______________________________________

**Zona de trabajo**
```sql
WITH
u AS (
    SELECT __________________
    FROM __________________
),
grid AS (
    SELECT __________________ AS week_offset
),
retention AS (
    SELECT
        __________________,
        __________________,
        COUNT(DISTINCT CASE
            WHEN EXISTS (
                SELECT 1
                FROM __________________
                WHERE __________________
            ) THEN __________________
        END) AS retained_users,
        COUNT(DISTINCT __________________) AS cohort_users
    FROM __________________
    GROUP BY __________________
)
SELECT
    __________________,
    __________________,
    ROUND(100.0 * __________________ / NULLIF(__________________, 0), 2) AS retention_pct
FROM retention
ORDER BY __________________;
```

**Notas**
¿Qué parte te resultó más difícil?
____________________________________________________________

### Ejercicio C-3: Tabla pivote de retención para heatmap (Nivel: ⭐⭐⭐ Avanzado)

**Consigna:** Transforma la tabla anterior a formato ancho: una fila por cohort y una columna por semana (`w0`, `w1`, `w2`, `w3`).

### Técnica clave
Completa:

- SQL no siempre tiene `PIVOT`, por eso usamos ___________________________
- La lógica general es `MAX(CASE WHEN week_offset = N THEN ... END)`

**Plantilla**
```sql
WITH ... -- reutiliza la lógica anterior
SELECT
    cohort_month,
    MAX(CASE WHEN week_offset = 0 THEN __________________ END) AS w0,
    MAX(CASE WHEN week_offset = 1 THEN __________________ END) AS w1,
    MAX(CASE WHEN week_offset = 2 THEN __________________ END) AS w2,
    MAX(CASE WHEN week_offset = 3 THEN __________________ END) AS w3
FROM __________________
GROUP BY __________________
ORDER BY __________________;
```

**Lectura**
| Cohort | ¿Se desvanece rápido o lento? | Observación |
|---|---|---|
| | | |
| | | |
| | | |

### Ejercicio C-4: Retención segmentada por plan (Nivel: ⭐⭐⭐ Avanzado)

**Consigna:** Extiende la tabla de retención para segmentarla por `plan` (`free` vs `paid`).

### Antes de escribir
- Columna extra a incluir en la CTE `u`: ______________________
- Nuevos `GROUP BY`: _________________________________________
- Pregunta de negocio que quiero responder: ____________________

**Borrador**
```sql
WITH
u AS (
    SELECT
        user_id,
        DATE_TRUNC('month', signup_ts) AS cohort_month,
        __________________
    FROM users
),
grid AS (
    SELECT UNNEST(ARRAY[0,1,2,3]) AS week_offset
),
retention AS (
    SELECT
        __________________,
        __________________,
        __________________,
        COUNT(DISTINCT CASE WHEN EXISTS ( ... ) THEN __________________ END) AS retained_users,
        COUNT(DISTINCT __________________) AS cohort_users
    FROM __________________
    GROUP BY __________________
)
SELECT
    cohort_month,
    plan,
    week_offset,
    ROUND(100.0 * retained_users / NULLIF(cohort_users, 0), 2) AS retention_pct
FROM retention
ORDER BY cohort_month, plan, week_offset;
```

**Interpretación**
¿Quién parece retener mejor: `free` o `paid`? ¿En qué semana se nota más?
____________________________________________________________

---
# Bloque 5 · Ejercicios integradores (15 min)

Estos ejercicios combinan todo lo visto: Data Journey, CTEs, Funnels y Cohorts.

### Ejercicio I-1: Journey individual de un usuario (Nivel: ⭐ Básico)

**Consigna:** Muestra el journey completo del usuario 1 y calcula el tiempo transcurrido entre cada evento y el anterior.

### Pistas conceptuales
- Función para mirar el evento anterior: ______________________
- Partición recomendada: _____________________________________
- Diferencia de tiempo en minutos: _____________________________

**Plantilla**
```sql
WITH journey AS (
    SELECT
        __________________,
        __________________,
        __________________,
        LAG(__________________) OVER (
            PARTITION BY __________________
            ORDER BY __________________
        ) AS prev_event_ts
    FROM __________________
    WHERE __________________
)
SELECT
    __________________,
    __________________,
    __________________,
    __________________,
    DATEDIFF('minute', __________________, __________________) AS minutos_desde_anterior
FROM journey
ORDER BY __________________;
```

**Lectura**
¿Qué parte del journey fue más rápida y cuál más lenta?
____________________________________________________________

### Ejercicio I-2: Funnel por canal de adquisición (Nivel: ⭐⭐ Intermedio)

**Consigna:** Construye un funnel `session_start → purchase` agrupado por `channel`.

### Diseña la solución
| Componente | Respuesta |
|---|---|
| Tablas que debo cruzar | |
| Campo de unión | |
| Métricas por canal | |
| Fórmula de conversión | |

**Borrador**
```sql
WITH base AS (
    SELECT
        __________________,
        __________________,
        __________________
    FROM __________________
    INNER JOIN __________________
        ON __________________
),
por_canal AS (
    SELECT
        __________________,
        COUNT(DISTINCT CASE WHEN __________________ THEN __________________ END) AS session_users,
        COUNT(DISTINCT CASE WHEN __________________ THEN __________________ END) AS purchase_users
    FROM base
    GROUP BY __________________
)
SELECT
    __________________,
    __________________,
    __________________,
    ROUND(100.0 * __________________ / NULLIF(__________________, 0), 2) AS conversion_rate_pct
FROM __________________
ORDER BY __________________ DESC;
```

**Conclusión**
¿Qué canal convierte mejor? ______________________________  
¿Qué canal convertiría peor? _____________________________

### Ejercicio I-3: Usuarios que abandonaron vs. completaron (Nivel: ⭐⭐⭐ Avanzado)

**Consigna:** Clasifica a cada usuario según su etapa máxima del journey:
- `Solo sesión`
- `Exploró`
- `Carrito`
- `Checkout`
- `Comprador`

### Estrategia
1. Para cada usuario, crear indicadores por etapa.
2. Determinar la etapa máxima alcanzada.
3. Contar cuántos usuarios hay por categoría.

**Mapa lógico**
| Indicador | ¿Cómo lo construiría? |
|---|---|
| `tiene_session` | |
| `tiene_view` | |
| `tiene_cart` | |
| `tiene_checkout` | |
| `tiene_purchase` | |

**Borrador**
```sql
WITH etapas_usuario AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_name = 'session_start' THEN 1 ELSE 0 END) AS tiene_session,
        MAX(CASE WHEN event_name = 'view_item' THEN 1 ELSE 0 END) AS tiene_view,
        MAX(CASE WHEN event_name = 'add_to_cart' THEN 1 ELSE 0 END) AS tiene_cart,
        MAX(CASE WHEN event_name = 'begin_checkout' THEN 1 ELSE 0 END) AS tiene_checkout,
        MAX(CASE WHEN event_name = 'purchase' THEN 1 ELSE 0 END) AS tiene_purchase
    FROM events
    GROUP BY user_id
),
clasificacion AS (
    SELECT
        user_id,
        CASE
            WHEN __________________ THEN 'Comprador'
            WHEN __________________ THEN 'Checkout'
            WHEN __________________ THEN 'Carrito'
            WHEN __________________ THEN 'Exploró'
            ELSE 'Solo sesión'
        END AS categoria_journey
    FROM __________________
)
SELECT
    __________________,
    COUNT(*) AS num_usuarios
FROM __________________
GROUP BY __________________
ORDER BY __________________;
```

**Análisis**
¿Cuál es la categoría más frecuente? ____________________________  
¿Qué te diría eso sobre el producto o el proceso? __________________

---
# Cierre (5 min)

## Takeaways de la sesión

Completa al final:

1. El **Data Journey** me ayuda a ______________________________________
2. Las **CTEs** son útiles porque ______________________________________
3. Un **funnel** sirve para ____________________________________________
4. Un **cohort** sirve para ____________________________________________
5. Antes de analizar siempre debo validar _______________________________

## Autoevaluación rápida
| Pregunta | Mi respuesta |
|---|---|
| Lo que mejor entendí hoy fue... | |
| Lo que todavía me cuesta es... | |
| El ejercicio más retador fue... | |
| Quiero repasar especialmente... | |

## Siguientes pasos

- **Próxima sesión:** práctica de construcción de Funnels en SQL.
- **Repaso recomendado:** vuelve a escribir al menos una consulta con 3 CTEs encadenadas.
- **Aplicación:** piensa en un producto real que uses y dibuja su funnel en 4 o 5 etapas.
- **Extensión opcional:** lleva la tabla pivote de retención a Google Sheets y conviértela en heatmap.

### ¿Necesitas ayuda?
Recuerda los canales de ayuda que tenemos disponibles:
- `DATA-CO-LEARNING`: Revisa los horarios de atencion en la guia dele studiante. Recuerda que tenemos horarios de apoyo todos los días para tus dudas puntuales!
- `DA_CONSULTA`: Uuedes publicar tus preguntas sobre el contenido de la plataforma o el proyecto 24/7. Recuerda que en tu pregunta debes de agregar el tag @Dataconsulta.
- `SPRINT FOCUS`: Para los Sprints 1 al 5 tenemos sesiones extra donde abordamos temáticas de cada sprint a rpofundidad. Revisa la guia del estudiante para conocer los horarios!
- `SESIONES 1:1`: Recuerda que puedes agendar sesiones 1:1 con un tutor. Agendalas con antelación y resuelve todas tus dudas, desde temás del proyecto, curso, carrera hasta problemas técnicos.
- `CANAL DE DISCORD DE COMMUNITY`: Recuerda que tu cohorte tiene un canal especial donde puedes compartir cualquier item interesante que quieras mostrar a tus compañeros de curso.

## Resumen de ejercicios

| # | Ejercicio | Nivel | Estado |
|---|-----------|-------|--------|
| CTE-1 | Tu primera CTE | ⭐ | ☐ |
| CTE-2 | Dos CTEs encadenadas | ⭐ | ☐ |
| CTE-3 | CTE que referencia otra CTE | ⭐⭐ | ☐ |
| CTE-4 | Reutilizar una CTE | ⭐⭐ | ☐ |
| CTE-5 | Primer evento por usuario | ⭐⭐⭐ | ☐ |
| F-1 | Funnel básico | ⭐ | ☐ |
| F-2 | Funnel con drop-off | ⭐⭐ | ☐ |
| F-3 | Funnel por mes | ⭐⭐ | ☐ |
| C-1 | Cohorts mensuales | ⭐ | ☐ |
| C-2 | Retención semanal | ⭐⭐⭐ | ☐ |
| C-3 | Pivote para heatmap | ⭐⭐⭐ | ☐ |
| C-4 | Retención segmentada por plan | ⭐⭐⭐ | ☐ |
| I-1 | Journey individual | ⭐ | ☐ |
| I-2 | Funnel por canal | ⭐⭐ | ☐ |
| I-3 | Clasificación de usuarios | ⭐⭐⭐ | ☐ |

### Espacio final de notas
____________________________________________________________  
____________________________________________________________  
____________________________________________________________